# Hallucination Detection — Basic Output Validation

## Problem Statement
Build a deterministic validator that catches LLM hallucinations in ETL error parsing before they reach downstream systems.

## My Approach
3 validation layers, applied in sequence — fail fast at each stage:
- **Layer 1 — Format:** `json.loads()` catches malformed JSON before anything else runs
- **Layer 2 — Schema:** Pydantic `SSISErrorRecord` validates field presence and types; `severity` must be `LOW/MED/HIGH`, optional fields default to `None`
- **Layer 3 — Grounding:** For extractable fields (`error_code`, `affected_table`, `pipeline_name`), check if the value literally exists in the original error string — if not, it's invented
`summary` and `severity` are excluded from grounding since they're model-generated judgments, not extracted facts.

## What I Learned
- **LLMs hallucinate confidently** — Message 3 still returned `0x80004002` on some runs. The model filled the gap rather than admitting absence.
- **`is_grounded()` is a simple but effective first filter** — substring check catches pure fabrications like invented hex codes or pipeline names not in the input.
- **Pydantic's `Optional[str]`** doesn't accept the string `"None"` — had to handle LLM returning the word `"None"` vs JSON `null` separately.
- **Prompt matters for hallucination rate** — adding `"IMPORTANT: return null if not present. Do NOT invent values."` reduced hallucinations on Messages noticeably.
- **`is_grounded` can't catch semantic errors** — Message 3 has two tables; if the model picks the wrong one, both strings exist in input so it passes. Deterministic checks have a ceiling.

## Where I Got Stuck
- LLM was returning the string `"None"` instead of JSON `null` for missing fields - Pydantic rejected it as an invalid type for `Optional[str]`. Fixed by adding a soft-match in `is_grounded()` that treats `"none"`, `"null"`, `"n/a"`, `"unknown"` as valid abstentions.

## What I'd Do Differently
- **Sanitize before Pydantic** — strip string `"None"`/`"null"` values to actual `None` before schema validation, rather than handling it inside `is_grounded`. Cleaner separation of concerns.
- **Use a different few-shot example** — the example in the prompt is Message 1 verbatim. The model can pattern-match the answer instead of reasoning. A different example would test extraction more honestly.


In [1]:
from groq import Groq
from pydantic import BaseModel, validator, ValidationError
from typing import Optional, Literal
import os
import json
from dotenv import load_dotenv


load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
MODEL = "llama-3.1-8b-instant"  

## My Solution (Naive)
*First attempt — written before reviewing feedback*

In [2]:

ERROR_MESSAGES = [
    "[2024-01-15 02:33:11] SSIS Package 'Load_Inventory_DW.dtsx' failed. Component 'OLE DB Destination' error on table [dbo].[fact_inventory]. Error code: 0xC0202009. Rows rejected: 412.",
    "[2024-01-15 03:11:45] Pipeline 'Customer_Churn_ETL' timed out after 3600s. Last checkpoint: transform_step_3. No rows written.",
    "[2024-01-15 04:02:18] Constraint violation in staging. Source table [dbo].[stg_orders] attempted insert into [dbo].[fact_orders]. FK constraint on customer_id failed. Violation count: 87.",
    "[2024-01-15 05:55:00] Error Error Error: NULL NULL NULL pipeline=??? table=UNKNOWN code=???",
]


In [3]:
#Schema Definition
class SSISErrorRecord(BaseModel):
    error_code: Optional[str] = None  
    affected_table: Optional[str] = None
    pipeline_name: Optional[str] = None
    severity: Literal["LOW", "MED", "HIGH"]
    summary: str

exclude_from_hallucination_check = {'summary' , 'severity'}


In [ ]:
def is_grounded(value, original_msg):
    if value is None:
        return True   
    if str(value).lower() in ["none", "null", "n/a", "unknown"]:
        return True
    return str(value) in original_msg  

def hallucinate_check (response_dict : dict , original_message : str) -> dict :
        output = {
        'response': response_dict,
        'validation': {
            'status': 'VALID',
            'errors': []
            }
        }
        for (key, value) in response_dict.items():
            if key not in  exclude_from_hallucination_check :
                if not is_grounded(value , original_message) :
                    output['validation']['status'] = 'GROUNDING_ERROR'
                    output['validation']['errors'].append({
                        'hallucinated_field': key,
                        'invented_value': value
                    })
        return output

def validate_response(response , original_message : str) -> dict | None :
    try:
        response_json = json.loads(response)  
    except json.JSONDecodeError as e:
        return {"status": "FORMAT_ERROR", "detail": str(e), "hallucinated_fields": []}

    try :
        response_dict= SSISErrorRecord(**response_json).model_dump()
    except ValidationError as e:
        missing = [err["loc"][0] for err in e.errors() if err["type"] == "missing"]
        wrong   = [err["loc"][0] for err in e.errors() if err["type"] != "missing"]
        return {
            "status": "SCHEMA_ERROR",
            "missing_fields": missing,
            "type_errors": wrong,
            "hallucinated_fields": []
        }
    return hallucinate_check(response_dict , original_message)

def parse_ssis_error(error_text: str , promptbuilder) -> json:
    prompt=promptbuilder(error_text)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a senior DBA specializing in SSIS pipeline triage."},
            {"role": "user", "content": prompt}
        ],
        response_format={"type": "json_object"}
    )
    return validate_response(response.choices[0].message.content , error_text)
    

In [5]:
def few_shot_prompt(error: str) -> str:
    return f"""Convert the SSIS error below into a structured JSON object.

Example:
Input: [2024-01-15 02:33:11] SSIS Package 'Load_Inventory_DW.dtsx' failed.Component 'OLE DB Destination' error on table [dbo].[fact_inventory].Error code: 0xC0202009. Rows rejected: 412.
Output: {{"error_code": "0xC0202009","affected_table": "fact_inventory","pipeline_name": "Load_Inventory_DW", "severity": "HIGH","summary": "SSIS package Load_Inventory_DW failed with OLE DB error on fact_inventory, rejecting 412 rows."}}

Schema rules:
error_code: str - code of error : hex code only (e.g. 0xC0202009)  , NOTE : return None if the information is not present in the error text.
affected_table: str - table which is affected by failure , NOTE : return None if the information is not present in the error text.
pipeline_name: str - pipeline name which is affected by failure , NOTE : return None if the information is not present in the error text.
severity: one of 'LOW', 'MED', 'HIGH' - based on error impact 
summary: str - 1 sentence human summary

IMPORTANT: If a value cannot be found in the error text, return null. Do NOT invent values.
Error: {error}
"""

In [ ]:
for index, error in enumerate(ERROR_MESSAGES):
    response = parse_ssis_error(error, few_shot_prompt)
    
    if response:
        response['validation']['message_id'] = index + 1
        print('Error message : ' , error)
        print()
        print(json.dumps(response['response']))
        print()
        val = response['validation']
        base_info = f" message_id={index+1} | status={val['status']}"
        
        if not val['errors']:
            print(f"{base_info} | hallucinated_field=NONE")
        else:
            # Loop through each error found in this specific message
            for err in val['errors']:
                print(f"{base_info} | hallucinated_field={err['hallucinated_field']} | invented_value=\"{err['invented_value']}\"")
        
        print("-" * 20)

Error message :  [2024-01-15 02:33:11] SSIS Package 'Load_Inventory_DW.dtsx' failed. Component 'OLE DB Destination' error on table [dbo].[fact_inventory]. Error code: 0xC0202009. Rows rejected: 412.

{"error_code": "0xC0202009", "affected_table": "fact_inventory", "pipeline_name": "Load_Inventory_DW", "severity": "HIGH", "summary": "SSIS package Load_Inventory_DW failed with OLE DB error on fact_inventory, rejecting 412 rows."}

 message_id=1 | status=VALID | hallucinated_field=NONE
--------------------
Error message :  [2024-01-15 03:11:45] Pipeline 'Customer_Churn_ETL' timed out after 3600s. Last checkpoint: transform_step_3. No rows written.

{"error_code": null, "affected_table": null, "pipeline_name": "Customer_Churn_ETL", "severity": "MED", "summary": "SSIS pipeline Customer_Churn_ETL timed out after 3600s, last checkpoint was in transform_step_3"}

 message_id=2 | status=VALID | hallucinated_field=NONE
--------------------
Error message :  [2024-01-15 04:02:18] Constraint violat

## Refactored Solution

In [8]:
def sanitize(d: dict) -> dict:
    """Convert string 'None'/'null'/'none' → actual None before Pydantic sees it."""
    return {
        k: (None if isinstance(v, str) and v.lower() in ("none", "null", "???") else v)
        for k, v in d.items()
    }

def is_grounded(value, original_msg):
    if value is None:
        return True   
    # Removed string check from here
    return str(value) in original_msg

def validate_response(response , original_message : str) -> dict | None :
    try:
        response_json = json.loads(response)  
    except json.JSONDecodeError as e:
        return {"status": "FORMAT_ERROR", "detail": str(e), "hallucinated_fields": []}

    # Layer — sanitize string "None" before schema check
    response_json_parsed = sanitize(response_json)
    try :
        response_dict= SSISErrorRecord(**response_json_parsed).model_dump()
    except ValidationError as e:
        missing = [err["loc"][0] for err in e.errors() if err["type"] == "missing"]
        wrong   = [err["loc"][0] for err in e.errors() if err["type"] != "missing"]
        return {
            "status": "SCHEMA_ERROR",
            "missing_fields": missing,
            "type_errors": wrong,
            "hallucinated_fields": []
        }
    return hallucinate_check(response_dict , original_message)

In [11]:
def few_shot_prompt(error: str) -> str:
    return f"""Convert the SSIS error below into a structured JSON object.

Example:
Input: [2024-01-15 02:33:11] SSIS Package 'Load_Inventory_DW.dtsx' failed.Component 'OLE DB Destination' error on table [dbo].[fact_inventory].Error code: 0xC0202009. Rows rejected: 412.
Output: {{"error_code": "0xC0202009","affected_table": "fact_inventory","pipeline_name": "Load_Inventory_DW", "severity": "HIGH","summary": "SSIS package Load_Inventory_DW failed with OLE DB error on fact_inventory, rejecting 412 rows."}}

Input :  [2024-01-15 03:11:45] Pipeline 'Customer_Churn_ETL' timed out after 3600s. Last checkpoint: transform_step_3. No rows written.
Output : {{"error_code": None, "affected_table": None, "pipeline_name": "Customer_Churn_ETL", "severity": "MED", "summary": "SSIS pipeline Customer_Churn_ETL timed out after 3600s, last checkpoint was in transform_step_3"}}

Schema rules:
error_code: str - code of error : hex code only (e.g. 0xC0202009)  , NOTE : return None if the information is not present in the error text.
affected_table: str - table which is affected by failure , NOTE : return None if the information is not present in the error text.
pipeline_name: str - pipeline name which is affected by failure , NOTE : return None if the information is not present in the error text.
severity: one of 'LOW', 'MED', 'HIGH' - based on error impact 
summary: str - 1 sentence human summary

IMPORTANT: If a value cannot be found in the error text, return null. Do NOT invent values.
Error: {error}
"""

In [13]:
for index, error in enumerate(ERROR_MESSAGES):
    response = parse_ssis_error(error, few_shot_prompt)
    
    if response:
        response['validation']['message_id'] = index + 1
        print('Error message : ' , error)
        print()
        print(json.dumps(response['response']))
        print()
        val = response['validation']
        base_info = f" message_id={index+1} | status={val['status']}"
        
        if not val['errors']:
            print(f"{base_info} | hallucinated_field=NONE")
        else:
            # Loop through each error found in this specific message
            for err in val['errors']:
                print(f"{base_info} | hallucinated_field={err['hallucinated_field']} | invented_value=\"{err['invented_value']}\"")
        
        print("-" * 20)

Error message :  [2024-01-15 02:33:11] SSIS Package 'Load_Inventory_DW.dtsx' failed. Component 'OLE DB Destination' error on table [dbo].[fact_inventory]. Error code: 0xC0202009. Rows rejected: 412.

{"error_code": "0xC0202009", "affected_table": "fact_inventory", "pipeline_name": "Load_Inventory_DW", "severity": "HIGH", "summary": "SSIS package Load_Inventory_DW failed with OLE DB error on fact_inventory, rejecting 412 rows."}

 message_id=1 | status=VALID | hallucinated_field=NONE
--------------------
Error message :  [2024-01-15 03:11:45] Pipeline 'Customer_Churn_ETL' timed out after 3600s. Last checkpoint: transform_step_3. No rows written.

{"error_code": null, "affected_table": null, "pipeline_name": "Customer_Churn_ETL", "severity": "MED", "summary": "SSIS pipeline Customer_Churn_ETL timed out after 3600s, last checkpoint was in transform_step_3"}

 message_id=2 | status=VALID | hallucinated_field=NONE
--------------------
Error message :  [2024-01-15 04:02:18] Constraint violat